# LAPORAN UJIAN TENGAH SEMESTER
## Sistem Akuisisi dan Pengelolaan Data

### Eksplorasi Data Pariwisata Bali: Destinasi Wisata, Informasi Cuaca, dan Data Daya Tarik Wisata

### I. Identitas Mahasiswa

| Atribut | Keterangan |
|---|---|
| Nama | Fransiska C. Oda |
| Program Studi | Informatika |
| NIM | 2501010235 |
| Angkatan | 2025 |
| Semester | 3 |
| Institusi | INSTIKI |
| Dosen Pengampu | Dr. I Gede Totok Suryawan, S.Kom., M.T. |

### II. Ruang Lingkup dan Metodologi

Proyek ini membahas akuisisi dan pengelolaan data pariwisata di Provinsi Bali. Pengumpulan data dilakukan menggunakan tiga metode, yaitu:

1. Open Data Download: dataset daya tarik wisata Bali dengan minimal 100 baris.
2. REST API: data cuaca Bali menggunakan Open-Meteo dengan minimal 30 baris.
3. Web Scraping: pengambilan daftar destinasi wisata dari halaman Dinas Pariwisata Provinsi Bali.

Data kemudian diperiksa, dibersihkan, dianalisis, dan digabungkan untuk menghasilkan informasi yang lebih terstruktur.

### III. Petunjuk Eksekusi

Jalankan seluruh sel notebook secara berurutan dari awal sampai akhir menggunakan menu Runtime → Run all. Pastikan semua sel berhasil dijalankan dan hasilnya tersimpan.

In [65]:
# 1. SETUP LINGKUNGAN DAN FUNGSI UTILITAS

import pandas as pd
import numpy as np
import json
import os
import requests
import time
import hashlib

from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Membuat folder proyek
os.makedirs("data/raw/download", exist_ok=True)
os.makedirs("data/raw/api", exist_ok=True)
os.makedirs("data/raw/scraping", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("docs", exist_ok=True)

print("Setup lingkungan berhasil!")
print("Folder proyek berhasil disiapkan.")

Setup lingkungan berhasil!
Folder proyek berhasil disiapkan.


In [66]:
file_path = "/content/2024-DATA-DTW-update-sesuai-SK-1.xlsx"

df = pd.read_excel(file_path)

print("Ukuran data awal:", df.shape)
display(df.head())

Ukuran data awal: (151, 12)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,DATA DAYA TARIK WISATA PROVINSI BALI TAHUN 2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,No,Kabupaten,Jenis DTW,NaN,NaN,NaN,NaN,NaN,Sesuai Standar,Belum Standar,Total\nKeseluruhan
4,NaN,NaN,NaN,Alam,NaN,Budaya,NaN,Buatan,NaN,NaN,NaN,NaN


In [67]:
import os

print(os.listdir("/content"))

['.config', 'UTS_Pariwisata_Bali_Lengkap.zip', 'data', 'data_cuaca_bali_31hari.json', 'proyek_uts_pariwisata_bali.zip', 'README.md', 'data_cuaca_bali_31hari.csv', '2024-DATA-DTW-update-sesuai-SK-1.xlsx', 'docs', 'sample_data']


In [68]:
# 3. MEMBERSIHKAN DATA DTW BALI

data_dtw = df.iloc[21:151, [2, 3, 6, 9]].copy()

data_dtw.columns = [
    "Kategori",
    "Kabupaten",
    "Nama_DTW",
    "Alamat"
]

data_dtw = data_dtw.dropna(subset=["Nama_DTW"])

data_dtw[["Kategori", "Kabupaten"]] = (
    data_dtw[["Kategori", "Kabupaten"]].ffill()
)

print("Ukuran data setelah dibersihkan:", data_dtw.shape)
display(data_dtw.head(10))

Ukuran data setelah dibersihkan: (102, 4)


,Kategori,Kabupaten,Nama_DTW,Alamat
21,Alam,Badung,Alas Pala (Sangeh),"Desa Sangeh, Kec. Abiansemal, Kab. Badung"
22,Alam,Badung,Bali Elephant Camp,"Desa Carangsari, Kec. Petang, Kab. Badung"
23,Alam,Badung,Pantai Kuta,"Kelurahan Kuta,Kec. Kuta"
24,Alam,Badung,Pantai Labuan Sait,"Desa Pecatu, Kec. Kuta Selatan, Kab. Badung"
25,Alam,Badung,Pantai Legian,"Kelurahan Legian, Kec. Kuta"
26,Alam,Badung,Pantai Melasti,"Desa Ungasan, Kec. Kuta Selatan"
27,Alam,Badung,Pantai Nusa Dua,"Kelurahan Tanjung Benoa, Kec. Kuta Selatan"
28,Alam,Badung,Pantai Pandawa,"Desa Kutuh, Kec. Kuta Selatan, Kab. Badung"
29,Alam,Badung,Pantai Tanjung Benoa,"Kelurahan Tanjung Benoa, Kec. Kuta Selatan"
30,Alam,Badung,Water Blow Peninsula,"Kelurahan Tanjung Benoa, Kec. Kuta Selatan"


In [69]:
# 4. MENYIMPAN DATA DTW YANG SUDAH BERSIH

data_dtw.to_csv(
    "data_dtw_bali_bersih.csv",
    index=False
)

print("Data DTW berhasil disimpan!")
print("Jumlah baris:", len(data_dtw))

Data DTW berhasil disimpan!
Jumlah baris: 102


In [70]:
# 5. PEMERIKSAAN KUALITAS DATA

print("Jumlah baris:", len(data_dtw))
print("\nJumlah data kosong:")
print(data_dtw.isnull().sum())

print("\nJumlah data duplikat:")
print(data_dtw.duplicated().sum())

print("\nTipe data:")
print(data_dtw.dtypes)

Jumlah baris: 102

Jumlah data kosong:
Kategori     0
Kabupaten    0
Nama_DTW     0
Alamat       0
dtype: int64

Jumlah data duplikat:
0

Tipe data:
Kategori     object
Kabupaten    object
Nama_DTW     object
Alamat       object
dtype: object


In [71]:
# 6. MENGAMBIL DATA CUACA DARI OPEN-METEO API

url_api = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": -8.65,
    "longitude": 115.22,
    "start_date": "2026-07-01",
"end_date": "2026-10-31",
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum"
    ],
    "timezone": "Asia/Makassar"
}

headers = {
    "User-Agent": "AcademicResearch/1.0"
}

response_api = requests.get(
    url_api,
    params=params,
    headers=headers,
    timeout=30
)

response_api.raise_for_status()

data_cuaca = response_api.json()

print("Pengambilan data API berhasil!")
print("Jumlah tanggal:", len(data_cuaca["daily"]["time"]))

Pengambilan data API berhasil!
Jumlah tanggal: 123


In [72]:
# 7. MENGUBAH DATA API MENJADI DATAFRAME

data_cuaca_df = pd.DataFrame({
    "Tanggal": data_cuaca["daily"]["time"],
    "Suhu_Maksimum": data_cuaca["daily"]["temperature_2m_max"],
    "Suhu_Minimum": data_cuaca["daily"]["temperature_2m_min"],
    "Curah_Hujan": data_cuaca["daily"]["precipitation_sum"]
})

data_cuaca_df["Tanggal"] = pd.to_datetime(data_cuaca_df["Tanggal"])

print("Ukuran data cuaca:", data_cuaca_df.shape)
display(data_cuaca_df.head())

Ukuran data cuaca: (123, 4)


,Tanggal,Suhu_Maksimum,Suhu_Minimum,Curah_Hujan
0,2026-07-01,28.5,20.4,0.0
1,2026-07-02,29.3,20.2,0.0
2,2026-07-03,29.5,20.7,0.2
3,2026-07-04,29.4,22.5,0.0
4,2026-07-05,29.3,21.6,0.0


In [73]:
# 8. MENYIMPAN DATA CUACA API

data_cuaca_df.to_csv(
    "data_cuaca_bali_31hari.csv",
    index=False
)

data_cuaca_df.to_json(
    "data_cuaca_bali_31hari.json",
    orient="records",
    indent=4
)

print("Data cuaca berhasil disimpan!")
print("Jumlah baris:", len(data_cuaca_df))

Data cuaca berhasil disimpan!
Jumlah baris: 123


## 9. WEB SCRAPING DATA PARIWISATA BALI

Pada tahap ini dilakukan pengambilan data nama daya tarik wisata dari website resmi Dinas Pariwisata Provinsi Bali menggunakan metode Web Scraping. Pengambilan data dilakukan menggunakan Python dan BeautifulSoup.

In [74]:
# 10. MENGAKSES WEBSITE DINAS PARIWISATA BALI

url_scraping = "https://disparda.baliprov.go.id/badung"

headers = {
    "User-Agent": "AcademicResearch/1.0 (email: kemahasiswa@instiki.ac.id)"
}

session = requests.Session()

retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)

time.sleep(2)

response = session.get(
    url_scraping,
    headers=headers,
    timeout=30
)

response.raise_for_status()

print("Status website:", response.status_code)
print("Panjang HTML:", len(response.text))

Status website: 200
Panjang HTML: 182283


In [75]:
# 11. MEMBACA ISI HTML MENGGUNAKAN BEAUTIFULSOUP

soup = BeautifulSoup(response.text, "html.parser")

judul = soup.find_all(["h1", "h2", "h3"])

print("Jumlah judul yang ditemukan:", len(judul))

for item in judul:
    print(item.get_text(strip=True))

Jumlah judul yang ditemukan: 17
Badung
Pura Uluwatu
Pantai Jimbaran
Pantai Kuta
Pantai Nusa Dua
Pantai Pandawa
Tirta Taman Mumbul
Water Blow Nusa Dua
Pantai Green Bowl
Sangeh Monkey Forest
Garuda Wisnu Kencana
Air Terjun Nungnung
TAMAN AYUN TEMPLE
Berita Terkini
Google Map
Pengaturan aksesibilitas
Persetujuan Cookie


In [76]:
# 12. MEMBERSIHKAN HASIL WEB SCRAPING

judul_wisata = [
    item.get_text(strip=True)
    for item in soup.find_all(["h1", "h2", "h3"])
]

data_scraping = pd.DataFrame({
    "Nama_Wisata": judul_wisata
})

data_scraping = data_scraping[
    ~data_scraping["Nama_Wisata"].isin([
        "Badung",
        "Berita Terkini",
        "Google Map",
        "Pengaturan aksesibilitas",
        "Persetujuan Cookie"
    ])
]

print("Jumlah data wisata:", len(data_scraping))
display(data_scraping)

Jumlah data wisata: 12


,Nama_Wisata
1,Pura Uluwatu
2,Pantai Jimbaran
3,Pantai Kuta
4,Pantai Nusa Dua
5,Pantai Pandawa
6,Tirta Taman Mumbul
7,Water Blow Nusa Dua
8,Pantai Green Bowl
9,Sangeh Monkey Forest
10,Garuda Wisnu Kencana


In [77]:
# 13. SCRAPING DATA DTW DARI 9 KABUPATEN/KOTA BALI

wilayah_bali = {
    "Badung": "badung",
    "Buleleng": "buleleng",
    "Bangli": "bangli",
    "Denpasar": "denpasar",
    "Gianyar": "gianyar",
    "Jembrana": "jembrana",
    "Karangasem": "karangasem",
    "Klungkung": "klungkung-potensi-wisata",
    "Tabanan": "tabanan"
}

daftar_wisata = []

judul_tidak_dipakai = [
    "Daya Tarik Wisata",
    "Berita Terkini",
    "Google Map",
    "Pengaturan aksesibilitas",
    "Persetujuan Cookie"
]

for kabupaten, halaman in wilayah_bali.items():

    url = "https://disparda.baliprov.go.id/" + halaman

    hasil = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    hasil.raise_for_status()

    soup_wisata = BeautifulSoup(
        hasil.text,
        "html.parser"
    )

    for item in soup_wisata.find_all("h3"):

        nama = item.get_text(strip=True)

        if nama and nama not in judul_tidak_dipakai:
            daftar_wisata.append({
                "Kabupaten": kabupaten,
                "Nama_Wisata": nama
            })

    time.sleep(2)

data_scraping_baru = pd.DataFrame(daftar_wisata)

data_scraping_baru = data_scraping_baru.drop_duplicates()

print("Jumlah data scraping:", len(data_scraping_baru))
display(data_scraping_baru)

Jumlah data scraping: 63


,Kabupaten,Nama_Wisata
0,Badung,Badung
1,Badung,Pura Uluwatu
2,Badung,Pantai Jimbaran
3,Badung,Pantai Kuta
4,Badung,Pantai Nusa Dua
...,...,...
58,Tabanan,Margarana Heroes Cemetery
59,Tabanan,ANGSERI HOT SPRING WATER
60,Tabanan,JATILUWIH TERACES
61,Tabanan,TANAH LOT TEMPLE


In [78]:
print(data_scraping_baru["Kabupaten"].value_counts())

Kabupaten
Badung        13
Karangasem    10
Bangli         7
Gianyar        7
Buleleng       6
Tabanan        6
Jembrana       6
Klungkung      5
Denpasar       3
Name: count, dtype: int64


In [79]:
# 14. MENGAMBIL DATA DARI HALAMAN ARSIP DTW BALI

arsip_wisata = []

for halaman in range(1, 6):
    url = "https://disparda.baliprov.go.id/daya-tarik-wisata"

    if halaman > 1:
        url += f"?page={halaman}"

    hasil = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    hasil.raise_for_status()

    soup_arsip = BeautifulSoup(
        hasil.text,
        "html.parser"
    )

    for item in soup_arsip.find_all("h3"):
        nama = item.get_text(strip=True)

        if nama and nama.lower() != "daya tarik wisata":
            arsip_wisata.append(nama)

    time.sleep(2)

data_arsip = pd.DataFrame({
    "Nama_Wisata": arsip_wisata
})

data_arsip = data_arsip.drop_duplicates()

print("Jumlah data dari halaman arsip:", len(data_arsip))
display(data_arsip)

Jumlah data dari halaman arsip: 59


,Nama_Wisata
0,Data Daya Tarik Wisata dan Desa Wisata Tahun 2...
1,Pura Uluwatu
2,Gunung Batur
3,Danau Batur
4,Pura Kehen
5,Pura Batur
6,Air Terjun Cepung
7,Trunyan Village
8,Pantai Jimbaran
9,AIR TERJUN GITGIT


In [80]:
# 15. MENGGABUNGKAN DAN MEMBERSIHKAN DATA SCRAPING

data_scraping_final = pd.concat(
    [
        data_scraping_baru[["Nama_Wisata"]],
        data_arsip[["Nama_Wisata"]]
    ],
    ignore_index=True
)

teks_tidak_valid = [
    "Badung",
    "Daya Tarik Wisata",
    "Berita Terkini",
    "Google Map",
    "Pengaturan aksesibilitas",
    "Persetujuan Cookie",
    "Pertanian",
    "Sawah yang Indah"
]

data_scraping_final["Nama_Wisata"] = (
    data_scraping_final["Nama_Wisata"].str.strip()
)

data_scraping_final = data_scraping_final[
    ~data_scraping_final["Nama_Wisata"].isin(teks_tidak_valid)
]

data_scraping_final = data_scraping_final.drop_duplicates(
    subset=["Nama_Wisata"]
).reset_index(drop=True)

print("Jumlah data scraping unik:", len(data_scraping_final))
display(data_scraping_final)

Jumlah data scraping unik: 63


,Nama_Wisata
0,Pura Uluwatu
1,Pantai Jimbaran
2,Pantai Kuta
3,Pantai Nusa Dua
4,Pantai Pandawa
...,...
58,ANGSERI HOT SPRING WATER
59,JATILUWIH TERACES
60,TANAH LOT TEMPLE
61,BERATAN LAKE TEMPLE


In [81]:
# 15. MENGGABUNGKAN DAN MEMBERSIHKAN DATA SCRAPING

# Menggabungkan hasil dari 9 kabupaten dan halaman arsip
data_scraping_final = pd.concat(
    [
        data_scraping_baru[["Nama_Wisata"]],
        data_arsip[["Nama_Wisata"]]
    ],
    ignore_index=True
)

# Menghapus teks yang bukan nama destinasi
teks_tidak_valid = [
    "Badung",
    "Daya Tarik Wisata",
    "Berita Terkini",
    "Google Map",
    "Pengaturan aksesibilitas",
    "Persetujuan Cookie",
    "Pertanian",
    "Sawah yang Indah"
]

data_scraping_final["Nama_Wisata"] = (
    data_scraping_final["Nama_Wisata"].str.strip()
)

data_scraping_final = data_scraping_final[
    ~data_scraping_final["Nama_Wisata"].isin(teks_tidak_valid)
]

# Menghapus nama yang benar-benar duplikat
data_scraping_final = data_scraping_final.drop_duplicates(
    subset=["Nama_Wisata"]
).reset_index(drop=True)

print("Jumlah data scraping unik:", len(data_scraping_final))
display(data_scraping_final)

Jumlah data scraping unik: 63


,Nama_Wisata
0,Pura Uluwatu
1,Pantai Jimbaran
2,Pantai Kuta
3,Pantai Nusa Dua
4,Pantai Pandawa
...,...
58,ANGSERI HOT SPRING WATER
59,JATILUWIH TERACES
60,TANAH LOT TEMPLE
61,BERATAN LAKE TEMPLE


In [ ]:
print("Jumlah data dari 9 kabupaten:", len(data_scraping_baru))
print("Jumlah data arsip:", len(data_arsip))
print("Jumlah data akhir:", len(data_scraping_final))

Jumlah data dari 9 kabupaten: 63
Jumlah data arsip: 59
Jumlah data akhir: 63


In [ ]:
display(data_scraping_final.to_string())

In [84]:
# Membersihkan nama kabupaten yang bukan destinasi wisata

nama_kabupaten = [
    "Badung", "Buleleng", "Bangli", "Denpasar",
    "Gianyar", "Jembrana", "Karangasem",
    "Klungkung", "Tabanan"
]

data_scraping_final = data_scraping_final[
    ~data_scraping_final["Nama_Wisata"].str.strip().str.lower().isin(
        [nama.lower() for nama in nama_kabupaten]
    )
].reset_index(drop=True)

print("Jumlah destinasi wisata setelah dibersihkan:", len(data_scraping_final))
display(data_scraping_final)

Jumlah destinasi wisata setelah dibersihkan: 55


,Nama_Wisata
0,Pura Uluwatu
1,Pantai Jimbaran
2,Pantai Kuta
3,Pantai Nusa Dua
4,Pantai Pandawa
5,Tirta Taman Mumbul
6,Water Blow Nusa Dua
7,Pantai Green Bowl
8,Sangeh Monkey Forest
9,Garuda Wisnu Kencana


In [85]:
data_scraping_final = data_scraping_final[
    ~data_scraping_final["Nama_Wisata"].str.contains(
        "Data Daya Tarik Wisata",
        case=False,
        na=False
    )
].reset_index(drop=True)

print("Jumlah data wisata yang valid:", len(data_scraping_final))
display(data_scraping_final)

Jumlah data wisata yang valid: 54


,Nama_Wisata
0,Pura Uluwatu
1,Pantai Jimbaran
2,Pantai Kuta
3,Pantai Nusa Dua
4,Pantai Pandawa
5,Tirta Taman Mumbul
6,Water Blow Nusa Dua
7,Pantai Green Bowl
8,Sangeh Monkey Forest
9,Garuda Wisnu Kencana


In [86]:
# 16. SCRAPING TAMBAHAN DESTINASI WISATA BALI

url_direktori = "https://www.direktoripariwisata.id/kawasan/prov/Bali"

hasil_direktori = requests.get(
    url_direktori,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

hasil_direktori.raise_for_status()

soup_direktori = BeautifulSoup(
    hasil_direktori.text,
    "html.parser"
)

daftar_direktori = []

for item in soup_direktori.find_all("li"):
    tautan = item.find("a", recursive=False)

    if tautan:
        nama = tautan.get_text(" ", strip=True)

        if nama:
            daftar_direktori.append({
                "Nama_Wisata": nama
            })

data_direktori = pd.DataFrame(daftar_direktori)

data_direktori = data_direktori.drop_duplicates()

print("Jumlah data dari Direktori:", len(data_direktori))
display(data_direktori)

Jumlah data dari Direktori: 242


,Nama_Wisata
0,Depan
1,Direktori
2,Wisata Alam
3,Wisata Buatan
4,Wisata Budaya
...,...
311,KPPN Bali Utara-Singaraja dan sekitarnya
312,KPPN Kuta-Sanur-Nusa Dua dan sekitarnya
313,KPPN Menjangan-Pemuteran dan sekitarnya
314,KPPN TN. Bali Barat dan sekitarnya


In [87]:
display(data_direktori.iloc[0:50])

,Nama_Wisata
0,Depan
1,Direktori
2,Wisata Alam
3,Wisata Buatan
4,Wisata Budaya
5,Taman Nasional
6,Destinasi Prioritas
7,Bromo
8,Candi Borobudur
9,Danau Toba


In [88]:
display(data_direktori.iloc[50:120])

,Nama_Wisata
51,GUNUNG KAWI SEBATU
52,GUNUNG KAWI TAMPAKSIRING
53,Pasar Sukawati
54,Sangeh Monkey Forest
55,Pantai Green Bowl
...,...
116,Air Terjun Singsing
117,Danau Buyan
118,Air Terjun Banyumala
119,Air Terjun Gitgit


In [89]:
display(data_direktori.iloc[120:190])

,Nama_Wisata
121,Air Terjun Aling - Aling
122,Air Terjun Colek Pamor
123,Air Terjun Campuhan/ Twin Waterfall
124,Air terjun Jembong
125,Air Terjun Kroya
...,...
186,Lembah Tukad Melangit
189,Monumen Puputan di Klungkung Bali
190,Lapangan Puputan Klungkung di Klungkung Bali
191,Pasar Seni Kota Semarapura di Klungkung Bali


In [90]:
display(data_direktori.iloc[190:242])

,Nama_Wisata
193,Museum Seni Lukis Klasik Nyoman Gunarsa di Klu...
194,Wisata Desa Kamasan di Klungkung Bali
195,Taman Gili Kerta Gosa di Klungkung Bali
196,Bajra Sandi Renon
197,Lapangan Puputan di Denpasar Barat Bali
198,Museum 3D I Am Bali di Panjer
199,Patung Titi Banda
200,Pasar Tradisional Badung
201,Pasar Seni Kumbasari di Pemecutan
202,Big Garden Corner Sanur


In [91]:
# 17. MEMBERSIHKAN DAN MENGGABUNGKAN DATA WISATA

# Mengambil data mulai dari posisi ke-30,
# karena 30 data awal berisi menu dan destinasi nasional
tambahan_bali = data_direktori.iloc[30:].copy()

# Menghapus data kawasan, bukan nama destinasi
tambahan_bali = tambahan_bali[
    ~tambahan_bali["Nama_Wisata"].str.contains(
        r"KSPN|KPPN|DPN BALI|Bali Barat",
        case=False,
        na=False
    )
]

# Menggabungkan dengan 54 data sebelumnya
data_gabungan_scraping = pd.concat(
    [
        data_scraping_final,
        tambahan_bali
    ],
    ignore_index=True
)

# Membersihkan spasi dan data kosong
data_gabungan_scraping["Nama_Wisata"] = (
    data_gabungan_scraping["Nama_Wisata"].str.strip()
)

data_gabungan_scraping = data_gabungan_scraping.dropna()

# Menghapus nama yang sama tanpa membedakan huruf besar/kecil
data_gabungan_scraping["kunci"] = (
    data_gabungan_scraping["Nama_Wisata"].str.lower()
)

data_gabungan_scraping = (
    data_gabungan_scraping.drop_duplicates(subset="kunci")
    .drop(columns="kunci")
    .reset_index(drop=True)
)

print("Jumlah data scraping gabungan:",
      len(data_gabungan_scraping))

display(data_gabungan_scraping)

Jumlah data scraping gabungan: 238


,Nama_Wisata
0,Pura Uluwatu
1,Pantai Jimbaran
2,Pantai Kuta
3,Pantai Nusa Dua
4,Pantai Pandawa
...,...
233,Festifal Omed-omedan di Sesetan Denpasar Bali
234,Dermaga Serangan di Pedungan Denpasar Bali
235,Museum Le Mayeur di Sanur Denpasar
236,Pasar Burung Satria di Denpasar Utara


In [92]:
print("Jumlah seluruh data:", len(data_gabungan_scraping))

print(
    "Jumlah data duplikat:",
    data_gabungan_scraping["Nama_Wisata"].duplicated().sum()
)

print(
    "Jumlah data kosong:",
    data_gabungan_scraping["Nama_Wisata"].isna().sum()
)

display(data_gabungan_scraping)

Jumlah seluruh data: 238
Jumlah data duplikat: 0
Jumlah data kosong: 0


,Nama_Wisata
0,Pura Uluwatu
1,Pantai Jimbaran
2,Pantai Kuta
3,Pantai Nusa Dua
4,Pantai Pandawa
...,...
233,Festifal Omed-omedan di Sesetan Denpasar Bali
234,Dermaga Serangan di Pedungan Denpasar Bali
235,Museum Le Mayeur di Sanur Denpasar
236,Pasar Burung Satria di Denpasar Utara


## 15. PENGGABUNGAN DATA (DATA INTEGRATION)

Pada tahap ini dilakukan penggabungan data antara sumber Download Excel dan Web Scraping berdasarkan nama daya tarik wisata. Proses penggabungan menggunakan metode Merge pada Pandas untuk menemukan data wisata yang memiliki nama sesuai pada kedua sumber.

In [93]:
# MENYIAPKAN KUNCI PENGGABUNGAN DATA

data_dtw = df.copy()

data_dtw["Kunci"] = (
    data_dtw["Nama_DTW"]
    .astype(str)
    .str.lower()
    .str.strip()
)

data_gabungan_scraping["Kunci"] = (
    data_gabungan_scraping["Nama_Wisata"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Kunci penggabungan berhasil dibuat!")

display(data_dtw[["Nama_DTW", "Kunci"]].head())

display(
    data_gabungan_scraping[
        ["Nama_Wisata", "Kunci"]
    ].head()
)

KeyError: 'Nama_DTW'

In [ ]:
print([nama for nama in globals() if "dtw" in nama.lower() or "download" in nama.lower()])

In [ ]:
import pandas as pd

file_path = "/content/2024-DATA-DTW-update-sesuai-SK-1.xlsx"
df = pd.read_excel(file_path)

print("Data berhasil dibaca!")
print("Ukuran data:", df.shape)
display(df.head())

In [ ]:
display(df.iloc[3:15])

In [ ]:
display(df.iloc[15:35])

In [ ]:
# MEMBERSIHKAN DATA DTW DARI EXCEL

data_dtw = df.iloc[21:, [2, 3, 6, 10]].copy()

data_dtw.columns = [
    "Kategori",
    "Kabupaten",
    "Nama_DTW",
    "Alamat"
]

data_dtw["Kategori"] = data_dtw["Kategori"].ffill()
data_dtw["Kabupaten"] = data_dtw["Kabupaten"].ffill()

data_dtw = data_dtw.dropna(subset=["Nama_DTW"])

data_dtw["Nama_DTW"] = data_dtw["Nama_DTW"].astype(str).str.strip()
data_dtw["Alamat"] = data_dtw["Alamat"].fillna("").astype(str).str.strip()

print("Jumlah data DTW:", len(data_dtw))
display(data_dtw)

In [ ]:
# MEMBUAT KUNCI DATA DTW

data_dtw["Kunci"] = (
    data_dtw["Nama_DTW"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Kunci DTW berhasil dibuat!")
display(data_dtw[["Nama_DTW", "Kunci"]].head())

In [ ]:
# MEMBUAT KUNCI DATA SCRAPING

data_gabungan_scraping["Kunci"] = (
    data_gabungan_scraping["Nama_Wisata"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("Kunci scraping berhasil dibuat!")
display(data_gabungan_scraping[["Nama_Wisata", "Kunci"]].head())

In [ ]:
# MENGGABUNGKAN DATA DTW DAN SCRAPING

data_hasil_merge = pd.merge(
    data_dtw,
    data_gabungan_scraping,
    on="Kunci",
    how="inner"
)

print("Penggabungan data berhasil!")
print("Jumlah data yang cocok:", len(data_hasil_merge))

display(data_hasil_merge)

In [ ]:
# PENGGABUNGAN DATA DTW DAN SCRAPING

data_hasil_merge = pd.merge(
    data_dtw,
    data_gabungan_scraping[["Kunci", "Nama_Wisata"]],
    on="Kunci",
    how="left"
)

print("Jumlah seluruh data:", len(data_hasil_merge))
print("Jumlah nama yang cocok:", data_hasil_merge["Nama_Wisata"].notna().sum())

display(data_hasil_merge)

In [ ]:
print("Jumlah baris:", len(data_hasil_merge))
print("\nData kosong:")
print(data_hasil_merge.isnull().sum())

print("\nJumlah duplikat:")
print(data_hasil_merge.duplicated().sum())

In [ ]:
print(data_hasil_merge.duplicated().sum())

In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)

data_hasil_merge.to_csv(
    "data/processed/data_wisata_gabungan_102.csv",
    index=False
)

print("Data berhasil disimpan!")

In [ ]:
import os

print(os.listdir("data/processed"))

In [ ]:
cek_file = pd.read_csv(
    "data/processed/data_wisata_gabungan_102.csv"
)

print("Jumlah baris:", len(cek_file))
display(cek_file.head(10))

In [ ]:
data_final = data_hasil_merge[
    ["Kategori", "Kabupaten", "Nama_DTW", "Alamat", "Nama_Wisata"]
].copy()

data_final["Alamat"] = data_final["Alamat"].fillna("")
data_final["Nama_Wisata"] = data_final["Nama_Wisata"].fillna("")

data_final.to_csv(
    "data/processed/data_wisata_gabungan_102.csv",
    index=False
)

print("Jumlah data akhir:", len(data_final))
display(data_final.head(10))

In [ ]:
print("DATA DTW:", len(data_dtw))
print("DATA SCRAPING:", len(data_gabungan_scraping))
print("DATA GABUNGAN:", len(data_final))

In [ ]:
print("DATA CUACA:", len(data_cuaca))

In [ ]:
import requests
import pandas as pd
import json
import os

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": -8.4095,
    "longitude": 115.1889,
    "start_date": "2026-07-01",
    "end_date": "2026-10-31",
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum"
    ],
    "timezone": "Asia/Makassar"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

hasil_cuaca = response.json()

data_cuaca = pd.DataFrame({
    "Tanggal": hasil_cuaca["daily"]["time"],
    "Suhu_Maksimum": hasil_cuaca["daily"]["temperature_2m_max"],
    "Suhu_Minimum": hasil_cuaca["daily"]["temperature_2m_min"],
    "Curah_Hujan": hasil_cuaca["daily"]["precipitation_sum"]
})

os.makedirs("data/raw/api", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

with open("data/raw/api/data_cuaca_bali.json", "w") as f:
    json.dump(hasil_cuaca, f, indent=4)

data_cuaca.to_csv(
    "data/processed/data_cuaca_bali.csv",
    index=False
)

print("Jumlah data cuaca:", len(data_cuaca))
display(data_cuaca.head())

In [ ]:
print("Jumlah baris:", len(data_cuaca))

print("\nData kosong:")
print(data_cuaca.isnull().sum())

print("\nJumlah duplikat:")
print(data_cuaca.duplicated().sum())

In [ ]:
data_cuaca.to_csv(
    "data/processed/data_cuaca_bali.csv",
    index=False
)

print("File cuaca berhasil disimpan!")
print(os.listdir("data/processed"))

In [ ]:
for folder in [
    "data/raw/download",
    "data/raw/api",
    "data/raw/scraping"
]:
    print("\nIsi folder:", folder)
    print(os.listdir(folder))

In [ ]:
import shutil

shutil.copy(
    "/content/2024-DATA-DTW-update-sesuai-SK-1.xlsx",
    "data/raw/download/2024-DATA-DTW-update-sesuai-SK-1.xlsx"
)

print(os.listdir("data/raw/download"))

In [ ]:
data_gabungan_scraping.to_csv(
    "data/raw/scraping/data_wisata_scraping.csv",
    index=False
)

print(os.listdir("data/raw/scraping"))

In [ ]:
for folder, subfolders, files in os.walk("data"):
    print(folder, ":", files)

In [ ]:
readme = """
# Eksplorasi Data Pariwisata Bali

## Deskripsi
Proyek ini mengumpulkan dan mengolah data pariwisata Bali dari tiga sumber:
1. Data DTW dari file Excel.
2. Data cuaca dari Open-Meteo API.
3. Data wisata melalui teknik web scraping.

## Jumlah Data
- Data DTW: 102 baris.
- Data scraping: 238 baris.
- Data cuaca: 123 baris.
- Data gabungan DTW dan scraping: 102 baris.
- Nama DTW yang cocok pada proses penggabungan: 24.

## Struktur Folder
- data/raw/download: data Excel asli.
- data/raw/api: data JSON dari API.
- data/raw/scraping: data hasil scraping.
- data/processed: data yang sudah diolah.

## Pengolahan Data
Data diperiksa untuk mengetahui data kosong dan duplikat.
Penggabungan DTW dan scraping menggunakan metode left join.

## Sumber Data
- Data DTW: Dinas Pariwisata Provinsi Bali.
- Data cuaca: Open-Meteo Archive API.
- Data scraping: sumber informasi pariwisata Bali.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README berhasil dibuat!")

In [ ]:
import os
import shutil

shutil.make_archive(
    "/content/proyek_uts_pariwisata_bali",
    "zip",
    "/content",
    "data"
)

print("ZIP berhasil dibuat!")
print(os.path.getsize("/content/proyek_uts_pariwisata_bali.zip"), "bytes")

In [ ]:
import zipfile
import os

with zipfile.ZipFile(
    "/content/UTS_Pariwisata_Bali_Lengkap.zip", "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    zipf.write("README.md", "README.md")

    for folder, subfolders, files in os.walk("data"):
        for file in files:
            path = os.path.join(folder, file)
            zipf.write(path, path)

print("ZIP lengkap berhasil dibuat!")

In [ ]:
with zipfile.ZipFile(
    "/content/UTS_Pariwisata_Bali_Lengkap.zip", "r"
) as zipf:
    for nama_file in zipf.namelist():
        print(nama_file)

In [ ]:
from google.colab import files

files.download("/content/UTS_Pariwisata_Bali_Lengkap.zip")